In [2]:
# Lab 7: Fine-Tuning BERT for Sentiment Classification
# 100% Correct Version (CPU/GPU Compatible)

# Install first:
# pip install transformers datasets accelerate -q

import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

# =========================
# DEVICE SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# =========================
# SAMPLE DATASET
# =========================
data = {
    "text": [
        "I love this product",
        "Worst experience ever",
        "Amazing service",
        "Very bad quality",
        "This movie is fantastic",
        "I hate this app"
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

dataset = Dataset.from_dict(data)

# =========================
# TOKENIZER
# =========================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

dataset = dataset.map(tokenize, batched=True)

# Remove unnecessary column
dataset = dataset.remove_columns(["text"])

# Rename label column
dataset = dataset.rename_column("label", "labels")

# Convert dataset to PyTorch tensors
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# =========================
# LOAD MODEL
# =========================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Move model to device
model.to(device)

# =========================
# TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# =========================
# TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# =========================
# TRAIN MODEL
# =========================
trainer.train()

# =========================
# PREDICTION
# =========================
text = "This movie is fantastic"

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=32
)

# Move tensors to SAME DEVICE
inputs = {key: value.to(device) for key, value in inputs.items()}

# Prediction
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

# =========================
# OUTPUT
# =========================
if prediction.item() == 1:
    print("Positive Sentiment")
else:
    print("Negative Sentiment")

Using Device: cuda


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,0.799008
2,0.869498
3,0.584529


Negative Sentiment
